# 02 · Canais do Wi-Fi

Escaneia as redes Wi-Fi vizinhas (macOS) e mostra quais canais estão menos congestionados,
para escolher o canal do roteador com dados em vez de no chute.

1. Rode o **scan instantâneo** para uma foto do momento
2. Rode o **monitoramento** (padrão 30 min) antes de mudar o canal do roteador: um scan só engana
3. Os arquivos vão para `output/` (`wifi_relatorio_*.txt` e `wifi_scans_*.csv`)

* Com o extra `wifi` (pyobjc/CoreWLAN) cada rede vem com o sinal real. Sem ele, o macOS
  esconde o sinal das vizinhas e a análise só conta redes.

A biblioteca está em `src/y_tools/wifi/`; aqui é só o playground.

In [ ]:
# Importações

import pandas as pd
from dotenv import load_dotenv

from y_tools.logger import setup_logger
from y_tools.paths import OUTPUT_DIR
from y_tools.wifi import LOGGER_NAME, Analise, Scanner, formatar_relatorio, Rede, monitorar, parsear_meus_canais

load_dotenv()
logger = setup_logger(LOGGER_NAME)

## Parâmetros

| Parâmetro | O que faz |
| --- | --- |
| `MINUTOS` | Duração do monitoramento (`0` = um scan só) |
| `INTERVALO` | Segundos entre scans |
| `MEUS_CANAIS` | Canal do roteador na banda em que o Mac **não** está conectado, ex.: `"2.4:11"`. O da rede conectada é detectado sozinho |
| `PROPRIO_DBM` | Sinal acima disso, no canal do roteador, é o seu roteador e fica de fora |
| `FORTE_DBM` | Sinal acima disso conta como vizinho forte |

In [ ]:
# Parâmetros

MINUTOS = 30
INTERVALO = 60
PROPRIO_DBM = -55
FORTE_DBM = -75
MEUS_CANAIS = "2.4:11"

scanner = Scanner()
print(f"Método: {scanner.metodo}")
rede_atual = scanner.rede_atual()
print(f"Conectado em: {rede_atual}")

# * A rede conectada é a que queremos melhorar: o canal dela vence o informado
meus_canais = {(b, c) for b, c in parsear_meus_canais(MEUS_CANAIS) if not rede_atual or b != rede_atual.banda}
if rede_atual:
    meus_canais.add((rede_atual.banda, rede_atual.canal))
print(f"Canais do seu roteador: {sorted(meus_canais)}")

## Scan instantâneo

Uma foto do momento: redes vizinhas por banda e canal.

In [ ]:
# Scan único

redes = scanner.escanear()
df = pd.DataFrame([r.__dict__ for r in redes]).sort_values(["banda", "canal", "rssi"], ascending=[True, True, False])
print(f"{len(df)} redes")
df.groupby(["banda", "canal"]).agg(redes=("rssi", "size"), sinal_max=("rssi", "max")).reset_index()

In [ ]:
# Análise do scan único

analise = Analise(proprio_dbm=PROPRIO_DBM, forte_dbm=FORTE_DBM, meus_canais=meus_canais)
analise.adicionar(redes)
print(formatar_relatorio(analise, scanner.metodo, minutos=0, rede_atual=rede_atual))

## Monitoramento

? Os vizinhos em canal automático mudam ao longo do dia. Vale rodar em horários diferentes
(noite costuma ser o pior) antes de decidir.

* Trava o kernel pelo tempo todo. Para deixar rodando em segundo plano, prefira a CLI:
  `python -m y_tools.wifi --minutos 30`

In [ ]:
# Monitoramento

resultado = monitorar(
    minutos=MINUTOS, intervalo=INTERVALO, proprio_dbm=PROPRIO_DBM, forte_dbm=FORTE_DBM,
    meus_canais=parsear_meus_canais(MEUS_CANAIS), scanner=scanner,
)
print(resultado.relatorio)

## Ler um CSV já gerado

Reabre os dados brutos de um monitoramento anterior (o mais recente em `output/`).

In [ ]:
# Último CSV

csvs = sorted(OUTPUT_DIR.glob("wifi_scans_*.csv"))
scans = pd.read_csv(csvs[-1]) if csvs else pd.DataFrame()
print(csvs[-1].name if csvs else "Nenhum CSV em output/")
scans.head()

In [ ]:
# Redes por canal ao longo dos scans

if not scans.empty:
    analise_csv = Analise(proprio_dbm=PROPRIO_DBM, meus_canais=meus_canais)
    eh_minha = [analise_csv.e_propria(Rede(int(r.canal), int(r.largura_mhz), str(r.banda), None if pd.isna(r.rssi_dbm) else int(r.rssi_dbm))) for r in scans.itertuples()]
    vizinhas = scans[[not m for m in eh_minha]]
    tabela = vizinhas.pivot_table(index="scan", columns=["banda", "canal"], values="rssi_dbm", aggfunc="size", fill_value=0)
    display(tabela.describe().loc[["mean", "min", "max"]].round(1))